# 09 - Wrap-up: selectors, orchestration and best practices

**You will learn**: tags and selectors, running the whole pipeline in the right order, scheduling with a Databricks job,
and a checklist of good practices.

In [ ]:
from helpers import *

## 1. Tags and selectors

You have used `--select` with names and tags. A few more:

| Selector | Meaning |
|---|---|
| `tag:gold` | all models tagged `gold` |
| `+fct_sales` / `fct_sales+` | upstream / downstream |
| `1+fct_sales` | only the direct parents |
| `path:models/silver` | a folder |
| `resource_type:seed` | one kind of resource |
| `source:bronze+` | everything downstream of a source |
| `state:modified+` | what changed vs a previous run (with `--state`), the basis of CI |
| `tag:gold,tag:daily` | **intersection** (comma); a space means union |

In [ ]:
dbt("ls --select tag:gold --resource-type model --output name")

In [ ]:
dbt("ls --select source:bronze.sales_orders+ --resource-type model --output name")

Named **selectors** live in a `selectors.yml` file, so a team shares one definition of "the nightly run".

In [ ]:
%%writefile ../project/selectors.yml
selectors:
  - name: marts
    description: Only the business marts.
    definition:
      method: fqn
      value: "alpsport.gold.mart_*"

  - name: nightly
    description: Everything that must be refreshed after a new bronze batch (snapshots, silver and gold).
    definition:
      union:
        - method: tag
          value: snapshot
        - method: tag
          value: silver
        - method: tag
          value: gold


In [ ]:
dbt("ls --selector marts --resource-type model --output name")

## 2. The full pipeline after a new batch

The correct order after new bronze data arrives:

1. `dbt snapshot`, to capture the changes of the dimensions (snapshots must see the source **before** it changes again);
2. `dbt build`, which runs seeds, models and tests in dependency order (`fct_sales` merges only the delta).

Your trainer loads one more batch for the whole class (same as in notebooks 06 and 07). Once it is announced, run the pipeline:

In [ ]:
batch_before = int(scalar("SELECT MAX(_batch_id) FROM bronze.sports_shop.sales_orders"))
print("last batch when you started this step:", batch_before)

In [ ]:
# Re-run this cell until your trainer has announced the new batch
status = bronze_status()
check("a new batch is available in bronze", int(status.last_batch[0]) > batch_before,
      "not yet: wait for your trainer's announcement, then re-run this cell")
status

In [ ]:
dbt("snapshot")
dbt("build --selector nightly")

Useful commands when something goes wrong:

| Command | Use |
|---|---|
| `dbt retry` | re-run only what failed or was skipped in the last invocation |
| `dbt build --select fct_sales --full-refresh` | rebuild an incremental model |
| `dbt clean` | delete `target/` and `dbt_packages/` |
| `dbt run-operation <macro>` | run a macro, e.g. maintenance |
| `dbt show --select stg_products --limit 5` | preview a model's result without materializing it |

In [ ]:
dbt("show --select stg_products --limit 5")

## 3. Schedule it with a Databricks job

Nobody wants to type `dbt build` every night. In this repository the file `resources/dbt-training.job.yml` defines a **Databricks job**
with a dbt task, deployed by **Declarative Automation Bundles** (DABs). Read it:

In [ ]:
print((TRAINING.parent / 'resources' / 'dbt-training.job.yml').read_text(encoding='utf-8'))

It runs `dbt deps`, `dbt seed`, `dbt snapshot` and `dbt build` with the profile in `dbt_profiles/profiles.yml` (credentials are injected by Databricks).
To deploy it from the repository root:

```bash
databricks bundle validate
databricks bundle deploy --target dev
databricks bundle run dbt-training_job
```

The `prod` target in `databricks.yml` deploys the same job as a production copy. A job can chain several tasks,
for example *generate/ingest data* then *dbt*.

## 4. What you built

In [ ]:
project_tree()

In [ ]:
dbt("ls --resource-type model --output name")
q(f'''
    SELECT table_catalog AS layer, COUNT(*) AS objects
    FROM system.information_schema.tables
    WHERE table_schema = '{SCHEMA}' AND table_catalog IN ('silver', 'gold')
    GROUP BY 1 ORDER BY 1
''')

## 5. Good practices checklist

**Structure**
- [ ] One layer per folder (`silver` / `gold`), one model per file, the name is the file name.
- [ ] Naming: `stg_` for staging, `dim_` / `fct_` for the star schema, `mart_` for business marts.
- [ ] Staging models only rename, cast, clean. Joins and business logic come later.
- [ ] Reference sources with `source()` and models with `ref()`, **never** hard-coded table names.

**Quality**
- [ ] Every primary key: `unique` + `not_null`. Every foreign key: `relationships`.
- [ ] Accepted values on categorical columns. Business rules as singular tests.
- [ ] `warn` for known raw-data problems, `error` for the layers you promise to be clean.
- [ ] Run `dbt build`, not `run` then `test`: bad data must not flow downstream.

**Performance and cost**
- [ ] Big tables: `incremental` with `unique_key`, and a periodic `--full-refresh`.
- [ ] Views for light transformations, tables for what is queried a lot.

**Team**
- [ ] Describe models and key columns; keep `docs generate` in CI.
- [ ] Use tags/selectors for scheduling; run `state:modified+` in pull requests.
- [ ] Separate dev and prod targets; never share a dev schema between people.

## 6. Check your knowledge

<details><summary>1. What is the difference between <code>ref()</code> and <code>source()</code>?</summary>
<code>source()</code> points to a table dbt does not build (raw data); <code>ref()</code> to another dbt resource (model, seed, snapshot). Both feed the DAG.
</details>

<details><summary>2. Why does <code>dbt build</code> skip models after a failed test?</summary>
To stop bad data flowing into downstream models; you fix upstream first.
</details>

<details><summary>3. An incremental model misses rows that arrive late with an old timestamp. What can you do?</summary>
Filter on a load timestamp (<code>_ingested_at</code>) or subtract a lookback window; and schedule a periodic <code>--full-refresh</code>.
</details>

<details><summary>4. Why snapshot a source and not a silver model?</summary>
The snapshot must record the raw state before any cleaning logic changes, and it must run before the source changes again.
</details>

<details><summary>5. Where do you write the query that checks "no order before the store opening date"?</summary>
A singular test: a <code>.sql</code> file in <code>tests/</code> returning the offending rows.
</details>

## Congratulations!

You built a complete bronze - silver - gold pipeline with sources, models, seeds, tests, incremental loading, snapshots and documentation.

**Next steps**: try unit tests (`unit_tests:` in YAML, dbt 1.8+), model contracts and versions, exposures for dashboards,
CI with `state:modified+`, and the semantic layer / Databricks metric views on top of the marts.